# 4. От углового решения к отклику в точке

Первая половина тетради относится к главе 6: разделение по числу столкновений,
пространственное обращение сферическими функциями Бесселя, два маршрута первого
порядка и радиальный кэш. Вторая половина — временные бины — принадлежит главе 7
и в этом проходе оставлена в прежнем, предварительном виде.


In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

from lighthit.angular import angular_components, dense_finite_rank_reference
from lighthit.cache import CacheGrid, ResponseCache, MOMENT_ORDERS
from lighthit.single import single_spectrum
from scipy.special import spherical_jn, eval_legendre


## 4.1. Разделение по числу столкновений

$G_0=D^{-1}$ — свободный резольвент, $\Gamma$ — диагональный оператор рассеяния.
$$h^{(0)}=G_0\mathbf 1,\qquad h^{(1)}=G_0\Gamma h^{(0)},\qquad
h^{(\ge2)}=(D-\Gamma)^{-1}\Gamma h^{(1)}.$$
В третьей формуле стоит **полный** резольвент, поэтому она суммирует все порядки
начиная со второго, а не только второй. Главное численно: компонента $\ge2$
считается от собственной правой части, а не вычитанием близких величин.


In [ ]:
L, k_probe = 8, 0.2
free, one, more = angular_components(np.array([k_probe]), 0., medium, L, L)
total = (free + one + more)[0]
control = dense_finite_rank_reference(k_probe, 0., medium, L, L, quadrature_order=1024)

print('h0 свободный  =', free[0, 0])
print('h0 один       =', one[0, 0])
print('h0 два и более=', more[0, 0])
print('|один|/|свободный|  =', abs(one[0, 0]/free[0, 0]))
print('|>=2|/|один|        =', abs(more[0, 0]/one[0, 0]))
rel = np.max(np.abs(total - control))/np.max(np.abs(control))
print('сумма против плотного контроля:', rel)
assert rel < 1e-10


## 4.2. Пространственное обращение

$$M_\ell(r,\omega)=i^\ell\sqrt{\tfrac{2\ell+1}{2}}
\int_0^\infty\frac{k^2dk}{2\pi^2}j_\ell(kr)h_\ell(k,\omega),
\qquad K=\sum_{\ell\le J}M_\ell P_\ell(\nu).$$
$j_\ell(kr)$ делает радиальное обращение, $P_\ell(\nu)$ возвращает угловую
зависимость уже после интеграла. Собираем $M_\ell$ руками по формуле и
сравниваем с компонентой решателя: это проверка нормировки $1/2\pi^2$,
множителя $\sqrt{(2\ell+1)/2}$ и фазы $i^\ell$.


In [ ]:
probe_settings = SolverSettings(8, 12, 3.0, 0.05, 8)
k_nodes, k_weights = probe_settings.quadrature()
J = probe_settings.spatial_degree
deg = np.arange(J + 1)
radius, nu = 20.0, 0.35
omega_probe = np.array([0.0, 0.05])

def multipoles(w, component):
    blocks = dict(zip(('free', 'one', 'more'),
                      angular_components(k_nodes, w, medium, 8, J)))
    phase = (1j)**deg*np.sqrt((2*deg + 1)/2)
    radial = spherical_jn(deg[None, :], k_nodes[:, None]*radius)
    weight = (k_weights*k_nodes**2)/(2*np.pi**2)
    return phase*np.einsum('k,kj,kj->j', weight, radial, blocks[component][:, :J+1])

displacement = radius*np.array([np.sqrt(1 - nu**2), 0., nu])
probe = PointGreenSolver(medium, probe_settings).solve(
    omega_probe, [displacement], direction=[0, 0, 1])
legendre = eval_legendre(deg, nu)
rebuilt = np.array([multipoles(float(w), 'more') @ legendre for w in omega_probe])
packaged = probe.components[:, 0, 2]
print('вручную :', rebuilt)
print('решатель:', packaged)
rel = np.max(np.abs(rebuilt - packaged))/np.max(np.abs(packaged))
print('относительное расхождение:', rel)
assert rel < 1e-12


## 4.3. Два маршрута первого порядка

`PointGreenSolver` собирает
$$K=K^{(0)}_{\text{аналитический}}+K^{(1)}_{\text{полная HG}}+K^{(\ge2)}_{L}.$$
Первый порядок он берёт **не** из угловой системы, а из координатной квадратуры
`single.single_spectrum` с полной индикатриссой и без обрезания по Лежандру.
Конечный-$L$ первый порядок существует и хранится в кэше.

Сравнивать их надо на изотропной геометрии, где выживает только $\ell=0$:
тогда сравнение изолирует угловой ранг среды от угловой разрешающей
способности выхода. Там маршруты сходятся. Для **направленного** точечного
источника и точечного приёмника первый порядок несёт интегрируемую особенность
$1/\rho$ в баллистическом корне, и мультипольный маршрут на разумных настройках
для него не сходится вовсе — это и есть причина, по которой production его там
не использует.


In [ ]:
# Изотропная вспышка, изотропный приёмник: выживает только l = 0.
def first_order_finite_L(radius, degree):
    s = SolverSettings(degree, 0, 6.0, 0.04, 10, True)
    k, kw = s.quadrature()
    _, first, _ = angular_components(k, 0., medium, degree, 0)
    weight = (kw*k*k)/(2*np.pi**2)
    return np.sqrt(0.5)*np.sum(weight*spherical_jn(0, k*radius)*first[:, 0])

for r_probe in (10., 20., 40.):
    exact, _ = single_spectrum(np.array([0.]), r_probe, None, medium)
    ratios = [first_order_finite_L(r_probe, L).real/exact[0].real
              for L in (4, 8, 16, 32)]
    print(f'r = {r_probe:5.1f} м  полная HG {exact[0].real:.6e}  '
          + '  '.join(f'L={L}: {v:.4f}' for L, v in zip((4, 8, 16, 32), ratios)))
    assert abs(ratios[2] - 1) < 1e-3

# Направленная геометрия: мультипольный первый порядок здесь не сходится.
directed = [ (multipoles(0.0, 'one') @ legendre).real ]
full_hg, _ = single_spectrum(np.array([0.0]), radius, nu, medium)
print(f'направленная геометрия, nu = {nu}: мультиполь {directed[0]:.3e}, '
      f'полная HG {full_hg[0].real:.3e} -- маршруты расходятся, и это ожидаемо')
print('решатель берёт полную HG:', probe.components[0, 0, 1].real)
assert abs(probe.components[0, 0, 1] - full_hg[0]) < 1e-15*abs(full_hg[0]) + 1e-30

# Баллистика: у направленной вспышки вне луча она равна нулю.
print('баллистика направленной вспышки:', probe.components[:, 0, 0])
iso = PointGreenSolver(medium, probe_settings).solve(
    omega_probe, [[0., 0., radius]], direction=None)
q0 = np.exp(-medium.extinction_per_m*radius)/(4*np.pi*radius**2)
analytic = q0*np.exp(1j*omega_probe*radius/medium.speed_m_per_ns)
print('изотропная баллистика против формулы:',
      np.max(np.abs(iso.components[:, 0, 0] - analytic)))
assert np.max(np.abs(iso.components[:, 0, 0] - analytic)) < 1e-18


## 4.4. Радиальный кэш

Кэш хранит $M_\ell(r,\omega)$ — не готовый ответ модуля. Радиусы
геометрические, интерполяция кубическая по $\log r$, перед ней снимается
масштаб $e^{-\mu_a r}/(4\pi r^2)$ (поглощение, не экстинкция). Баллистика не
хранится: в кэше только `one_finite_L` и `two_or_more`. Экстраполяция запрещена.


In [ ]:
grid = CacheGrid.geometric(5., 60., 16, omega_probe)
cache = ResponseCache.build(medium, probe_settings, grid)
print('оси порядков:', MOMENT_ORDERS, ' форма:', cache.moments.shape)

node = 8
r_node = float(grid.radii_m[node])

# Та же формула, что в 4.2, но на произвольном радиусе.
def multipoles_r(r, w, component):
    blocks = dict(zip(('free', 'one', 'more'),
                      angular_components(k_nodes, w, medium, 8, J)))
    phase = (1j)**deg*np.sqrt((2*deg + 1)/2)
    radial = spherical_jn(deg[None, :], k_nodes[:, None]*r)
    weight = (k_weights*k_nodes**2)/(2*np.pi**2)
    return phase*np.einsum('k,kj,kj->j', weight, radial, blocks[component][:, :J+1])

for axis, (name, component) in enumerate(zip(MOMENT_ORDERS, ('one', 'more'))):
    exact = np.array([multipoles_r(r_node, float(w), component) for w in omega_probe])
    rel = np.max(np.abs(cache.moments[:, node, :, axis] - exact))/np.max(np.abs(exact))
    print(f'узел сетки, {name}: {rel:.2e}')
    assert rel < 1e-12

r_off = 17.3
interp = cache.moments_at(np.array([r_off]))
for axis, (name, component) in enumerate(zip(MOMENT_ORDERS, ('one', 'more'))):
    exact = np.array([multipoles_r(r_off, float(w), component) for w in omega_probe])
    rel = np.max(np.abs(interp[:, 0, :, axis] - exact))/np.max(np.abs(exact))
    print(f'между узлами, {name}: {rel:.2e}  (это проверка интерполяции)')

for bad in (2.0, 200.0):
    try:
        cache.moments_at(np.array([bad]))
    except ValueError as error:
        print(f'r = {bad} м отвергнут:', error)
    else:
        raise AssertionError('экстраполяция должна быть запрещена')


## 4.5. Временные бины — материал главы 7

Ниже прежний, предварительный раздел о переходе от спектра к бинам. Он будет
переписан вместе с главой 7; здесь он оставлен работающим, но не финальным.


In [ ]:
settings=SolverSettings(24,120,4.,.05,8)
omega=np.linspace(0,1.2,241)
t0=time.perf_counter()
result=PointGreenSolver(medium,settings).solve(omega,[np.sqrt(300.),0.,10.],direction=[0,0,1])
print('Elapsed [s]:',time.perf_counter()-t0)
print('Charge [m^-2]:',result.charge_per_m2)
print('Stage timings:',result.timings_s)
fig,ax=plt.subplots();ax.plot(omega,result.components[:,0,:].sum(1).real,label='real');ax.plot(omega,result.components[:,0,:].sum(1).imag,label='imag');ax.set(xlabel='omega [rad/ns]',ylabel='spectrum [m^-2]');ax.legend();plt.show()

In [ ]:
front=result.front_time_ns[0]
edges=np.arange(front-20,front+401,2.)
raw=result.readout(edges,sigma_ns=0.)
blur=result.readout(edges,sigma_ns=3.)
print('Readout type:',type(raw))

## 4.3. Почему интегрируем по бинам

Для центра бина $t_b$ и ширины $\Delta t$:
$$\int_{t_b-\Delta t/2}^{t_b+\Delta t/2}e^{-i\omega t}dt
=\Delta t\,\operatorname{sinc}(\omega\Delta t/2)e^{-i\omega t_b}.$$
Аппаратное гауссовское размытие добавляет $e^{-\omega^2\sigma^2/2}$.
Заряд — значение спектра при нулевой частоте, не значение импульса в максимуме.

При равномерной сетке $\Delta\omega$ сумма имеет период $2\pi/\Delta\omega$.
Проверять причинность нужно до readout: гауссовское размытие само даёт
сигнал раньше исходного фронта.

In [ ]:
# TimeProfile keeps integrated bin charges, not samples of a density.
print('Array shape (receiver, bin, order):', raw.components.shape)
print('Raw diagnostics:', raw.diagnostics)
print('Readout diagnostics:', blur.diagnostics)
fig, ax = plt.subplots()
ax.plot(raw.centers_ns, raw.rate_per_m2_ns[0], label='No instrument smearing')
ax.plot(blur.centers_ns, blur.rate_per_m2_ns[0], label='Gaussian sigma = 3 ns')
ax.axvline(front, linestyle='--', label='Geometrical front')
ax.set(xlabel='Time [ns]', ylabel='Response [m^-2 ns^-1]')
ax.legend(); plt.show()
fig, ax = plt.subplots()
widths = np.diff(raw.edges_ns)
for order, label in enumerate(['Direct', 'One scattering', 'Two or more']):
    ax.plot(raw.centers_ns, raw.components[0, :, order] / widths, label=label)
ax.set(xlabel='Time [ns]', ylabel='Unsmeared component [m^-2 ns^-1]')
ax.legend(); plt.show()
period = 2*np.pi/(omega[1]-omega[0])
print('Period associated with frequency spacing [ns]:', period)
print('Integrated charge in this finite time window [m^-2]:', raw.values_per_m2.sum())
print('Charge from omega=0, whole time axis [m^-2]:', result.charge_per_m2)

## Задания

1. Показать, что $h^{(0)}+h^{(1)}+h^{(\ge2)}$ равно решению полной системы, и
оценить, сколько знаков потерялось бы при вычислении $\ge2$ вычитанием.
2. Провести угловой интеграл в выводе $M_\ell$ полностью и проверить константу
$1/2\pi^2$ и фазу $i^\ell$.
3. Убрать множитель $i^\ell$ из ручной сборки и посмотреть, что именно ломается —
модуль или фаза.
4. Менять по одному $L$, $J$, $k_{\max}$, ширину панели и порядок квадратуры;
показать, что это пять независимых источников ошибки.
5. Измельчить радиальную сетку кэша и проверить, что ошибка интерполяции падает,
а моменты на узлах не меняются.
6. Объяснить, почему в масштабе кэша стоит $\mu_a$, а не $\mu_t$.

Код: `angular.py`, `green.py`, `single.py`, `cache.py`; книга: глава 6.
